,
## 4.7生成文本

[返回 4.6 节](ch04_4.6.ipynb) | [返回 4.1–4.5 节](ch04.ipynb)

**独立运行准备：**下面的单元从 `gpt.py` 导入 GPT 模型，并初始化配置、分词器和随机权重。修改 4.6 节笔记本中的 `GPTModel` 不会自动更改 `gpt.py`。

In [1]:
import torch
import tiktoken
from gpt import GPTModel

GPT_CONFIG_124M = {
    "vocab_size": 50257,    # Vocabulary size
    "context_length": 1024, # Context length
    "emb_dim": 768,         # Embedding dimension
    "n_heads": 12,          # Number of attention heads
    "n_layers": 12,         # Number of layers
    "drop_rate": 0.1,       # Dropout rate
    "qkv_bias": False       # Query-Key-Value bias
}

tokenizer = tiktoken.get_encoding("gpt2")
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)

像我们上面实现的GPT模型这样的llm是用来一次生成一个单词的

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch04_compressed/16.webp" width="600px">

—下面的`generate_text_simple`函数实现贪婪解码，是一种简单快速的生成文本的方法
在贪婪解码中，在每一步中，模型选择概率最高的单词（或令牌）作为下一个输出（最高的logit对应最高的概率，所以我们技术上甚至不需要显式地计算softmax函数）
-在下一章中，我们将实现一个更高级的`generate_text`功能
下图描述了给定输入上下文的GPT模型如何生成下一个单词标记

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch04_compressed/17.webp" width="600px">

In [2]:
def generate_text_simple(model, idx, max_new_tokens, context_size):
    # idx is (batch, n_tokens) array of indices in the current context
    for _ in range(max_new_tokens):
        
        # Crop current context if it exceeds the supported context size
        # E.g., if LLM supports only 5 tokens, and the context size is 10
        # then only the last 5 tokens are used as context
        idx_cond = idx[:, -context_size:]
        
        # Get the predictions
        with torch.no_grad():
            logits = model(idx_cond)
        
        # Focus only on the last time step
        # (batch, n_tokens, vocab_size) becomes (batch, vocab_size)
        logits = logits[:, -1, :]  

        # Apply softmax to get probabilities
        probas = torch.softmax(logits, dim=-1)  # (batch, vocab_size)

        # Get the idx of the vocab entry with the highest probability value
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)  # (batch, 1)

        # Append sampled index to the running sequence
        idx = torch.cat((idx, idx_next), dim=1)  # (batch, n_tokens+1)

    return idx

上面的`generate_text_simple`实现了一个迭代过程，一次创建一个令牌<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch04_compressed/18.webp" width="600px">

-让我们准备一个输入示例：

In [3]:
start_context = "Hello, I am"

encoded = tokenizer.encode(start_context)
print("encoded:", encoded)

encoded_tensor = torch.tensor(encoded).unsqueeze(0)
print("encoded_tensor.shape:", encoded_tensor.shape)

encoded: [15496, 11, 314, 716]
encoded_tensor.shape: torch.Size([1, 4])


In [4]:
model.eval() # disable dropout

out = generate_text_simple(
    model=model,
    idx=encoded_tensor, 
    max_new_tokens=6, 
    context_size=GPT_CONFIG_124M["context_length"]
)

print("Output:", out)
print("Output length:", len(out[0]))

Output: tensor([[15496,    11,   314,   716, 27018, 24086, 47843, 30961, 42348,  7267]])
Output length: 10


- 移除批次维度并转换回文本：

In [5]:
decoded_text = tokenizer.decode(out.squeeze(0).tolist())
print(decoded_text)

Hello, I am Featureiman Byeswickattribute argue


"Hello, I am Featureiman Byeswickattribute argue"

- 请注意，该模型尚未经过训练；因此上述生成的文本是随机胡言乱语
- 我们将在下一章中对该模型进行训练

&nbsp;
## 总结与要点

- 请参阅 [./gpt.py](./gpt.py) 脚本，这是一个自包含的 Python 脚本，其中包含我们在本 Jupyter 笔记本中实现的完整 GPT 模型
- 课后习题解答可以在 [./exercise-solutions.ipynb](./exercise-solutions.ipynb) 中找到